> 在线讲义：[第 8 章 · π0：大脑 + 小脑，动作专家（action expert）](https://wikieden.github.io/vla-from-scratch/lessons/08-pi0.html) · 本书《VLA 从零到 π0》

# 第 8 章 · π0：大脑 + 小脑，动作专家（action expert）

配套讲义：仓库里 `vla-deep-dive/notes/lessons/08-pi0.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `LATENCY_MS` | 14 + 32 + 27 = 73 ms |
| 第 2 步 | `mix(1, −1, 0.3)`、`euler_path(...)` | −0.4；每步走 0.2，10 步从 −1 到 +1 |
| 第 3 步 | `block_mask(3, 1, 3)` | 前两块看不到动作 → K/V 可缓存 |
| 插问 2 | `attention_weights(q, ks)` | 训练前杯子权重 0%，训练后 91% |
| 第 4 步 | `tau_cdf(τ)` | [0, 0.1] 15%、[0, 0.5] 65%、[0.9, 1] 3% |

时间取自 pi0-2410.24164.txt 附录 D 表 I（RTX 4090、3 个相机）；τ 抽样公式取自附录 B。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
import math
LATENCY_MS = {"图像编码": 14, "大脑前缀前向（1 次）": 32, "小脑 flow 10 步（合计）": 27}

**函数 `mix`**：训练时造"半雾半清"的动作：A^τ = τA + (1 − τ)ε。

In [ ]:
def mix(a, eps, tau):
    """训练时造"半雾半清"的动作：A^τ = τA + (1 − τ)ε。"""
    return tau * a + (1 - tau) * eps

**函数 `euler_path`**：推理：从噪声出发，每步走 δ = 1/steps；velocity(x, τ) 给出速度。

In [ ]:
def euler_path(start, velocity, steps=10):
    """推理：从噪声出发，每步走 δ = 1/steps；velocity(x, τ) 给出速度。"""
    x, delta, path = start, 1 / steps, [start]
    for k in range(steps):
        x = x + delta * velocity(x, k * delta)
        path.append(x)
    return path

**函数 `block_mask`**：π0 的三块注意力：前缀只看前缀；状态看前缀 + 自己；动作看全部。

In [ ]:
def block_mask(n_prefix, n_state, n_action):
    """π0 的三块注意力：前缀只看前缀；状态看前缀 + 自己；动作看全部。"""
    n = n_prefix + n_state + n_action
    m = np.zeros((n, n), dtype=int)
    m[:n_prefix, :n_prefix] = 1
    m[n_prefix:n_prefix + n_state, :n_prefix + n_state] = 1
    m[n_prefix + n_state:, :] = 1
    return m

**函数 `attention_weights`**：2 维玩具：Q 和各个 K 点积（Q 长 3、K 长 1），再 softmax。

In [ ]:
def attention_weights(q_deg, k_degs, q_len=3.0):
    """2 维玩具：Q 和各个 K 点积（Q 长 3、K 长 1），再 softmax。"""
    scores = np.array([q_len * math.cos(math.radians(q_deg - k)) for k in k_degs])
    e = np.exp(scores - scores.max())
    return e / e.sum()

**函数 `tau_cdf`**：π0 的 τ 抽样：x = (s − τ)/s ~ Beta(1.5, 1)，P(τ ≤ t) = 1 − ((s − t)/s)^1.5。

In [ ]:
def tau_cdf(tau, s=0.999):
    """π0 的 τ 抽样：x = (s − τ)/s ~ Beta(1.5, 1)，P(τ ≤ t) = 1 − ((s − t)/s)^1.5。"""
    x = max(0.0, (s - tau) / s)
    return 1 - x**1.5

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 推理时间账")
for k, v in LATENCY_MS.items():
    print(f"  {k}：{v} ms")
print(f"  合计 {sum(LATENCY_MS.values())} ms")

In [ ]:
print("\n第二步 flow matching（一维：ε = −1，A = +1）")
a, eps = 1.0, -1.0
print(f"  τ = 0.3：A^τ = {mix(a, eps, 0.3):+.1f}，目标速度 u = A − ε = {a - eps:+.0f}")
path = euler_path(eps, lambda x, t: a - eps)
print(f"  每步都猜对速度：{[round(p, 1) for p in path]}")

In [ ]:
print("\n第三步 三块注意力（2 图 + 1 指令 | 1 状态 | 3 动作）")
m = block_mask(3, 1, 3)
print(m)
print(f"  前两块看得到动作吗：{bool(m[:4, 4:].any())} → K/V 可以缓存，大脑算 1 次、小脑算 10 次")

In [ ]:
print("\n插问 2 动作专家的 Q 怎么学会查大脑的 K")
ks = [25, 115, 205, 295]  # 杯子、碗、桌面、"放进"
for name, q in (("训练前（指向桌面）", 205), ("训练后（指向杯子）", 25)):
    w = attention_weights(q, ks)
    print(f"  {name}：杯子权重 {w[0]:.0%}")

In [ ]:
print("\n第四步 τ 怎么抽：多练雾大的地方")
for lo, hi in ((0, 0.1), (0, 0.5), (0.9, 1.0)):
    print(f"  τ 落在 [{lo}, {hi}]：均匀 {hi - lo:.0%}，π0 {tau_cdf(hi) - tau_cdf(lo):.0%}")

## 3. 代码里才看得到的

1. LeRobot 的 `euler_integrate` 默认约定是 `NOISE_AT_ONE`：噪声在 τ = 1、真动作在 τ = 0，和 π0 论文（τ = 0 是噪声）正好反过来；读源码时时间方向要倒过来看。
2. `tau_cdf` 用的是 Beta(1.5, 1) 的累积 $x^{1.5}$，再把 x = (s − τ)/s 换回 τ；τ > 0.999 时 x < 0，被截到 0。
3. `attention_weights` 把 256 维压成 2 维，只为了能画出来；真实 head 维度的点积还要除以 √d。

## 4. 自己改一改

- 把 `euler_path` 的步数改成 5，看每步走多远、终点还对不对。
- 把 Beta 的第一个参数从 1.5 改成 3，看 τ 小的那一端多抽多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 大脑看图读指令，小脑出动作；同一会议室开会，各用各的权重。
> 大脑算一次存起来，小脑问路走十步；雾里起点随机，走进哪座山都行。
> 三排座位：同排互相听，后排听前排；前排听不见后排，所以前排算一次就够。